# Chapter 10 - Kubernetes Homework

This notebook provides a structured approach to completing the Kubernetes homework.

**Author:** ML Zoomcamp 2025  
**Topic:** Deploying ML Models to Kubernetes  
**Model:** Bank Marketing Prediction

## Configuration Section
### 📝 Modify these parameters as needed

In [1]:
# ==================== CONFIGURATION ====================
# Modify these variables based on homework requirements

# Docker Configuration
DOCKER_IMAGE_NAME = "zoomcamp-model"
DOCKER_IMAGE_TAG = "3.13.10-hw10"
DOCKER_IMAGE_FULL = f"{DOCKER_IMAGE_NAME}:{DOCKER_IMAGE_TAG}"
DOCKER_HUB_IMAGE = "svizor/zoomcamp-model:3.13.10-hw10"  # Fallback if build fails

# Kubernetes Configuration
CLUSTER_NAME = "kind"
DEPLOYMENT_NAME = "subscription"
SERVICE_NAME = "subscription"
HPA_NAME = "subscription-hpa"
NAMESPACE = "default"

# Service Configuration
SERVICE_PORT = 80
TARGET_PORT = 9696
LOCAL_PORT = 9696

# Resource Configuration
MEMORY_REQUEST = "64Mi"
MEMORY_LIMIT = "128Mi"
CPU_REQUEST = "100m"
CPU_LIMIT = "200m"

# HPA Configuration
MIN_REPLICAS = 1
MAX_REPLICAS = 3
CPU_THRESHOLD = 20

# Test Data (Question 1 & 6)
TEST_CLIENT_DATA = {
    "job": "management",
    "duration": 400,
    "poutcome": "success"
}

# Load Test Configuration (Question 8)
LOAD_TEST_REQUESTS = 1000
LOAD_TEST_DELAY = 0.01  # seconds between requests

print("✅ Configuration loaded successfully!")
print(f"Docker Image: {DOCKER_IMAGE_FULL}")
print(f"Deployment: {DEPLOYMENT_NAME}")
print(f"Service: {SERVICE_NAME}")

✅ Configuration loaded successfully!
Docker Image: zoomcamp-model:3.13.10-hw10
Deployment: subscription
Service: subscription


## Question 1: Local Docker Testing

**Task:** Build and run the Docker container locally, then test the prediction endpoint.

**What to find:** The probability value returned by the model.

In [2]:
# Check if Docker is running
!docker --version

Docker version 29.0.1, build eedd969


In [3]:
# Option 1: Build the Docker image (if you have the Dockerfile)
# Uncomment if you want to build locally
# !docker build -t {DOCKER_IMAGE_FULL} .

# Option 2: Pull the pre-built image
# !docker pull {DOCKER_HUB_IMAGE}
# !docker tag {DOCKER_HUB_IMAGE} {DOCKER_IMAGE_FULL}

In [4]:
# Stop any existing container on port 9696
import subprocess
subprocess.run(["docker", "stop", "zoomcamp-test"], capture_output=True)
subprocess.run(["docker", "rm", "zoomcamp-test"], capture_output=True)

# Run the container in detached mode
!docker run -d --name zoomcamp-test -p {LOCAL_PORT}:9696 {DOCKER_IMAGE_FULL}

95bf1783fdca2f7c13d28c69ba447b519484014af13bead8d368be6487fb51b3


In [5]:
# Wait for container to be ready
import time
print("Waiting for container to start...")
time.sleep(3)

# Check container status
!docker ps | grep zoomcamp-test

Waiting for container to start...


95bf1783fdca   zoomcamp-model:3.13.10-hw10   "uvicorn q6_predict:…"   3 seconds ago   Up 3 seconds   0.0.0.0:9696->9696/tcp, [::]:9696->9696/tcp   zoomcamp-test


In [6]:
# Test the prediction endpoint
import requests
import json

url = f"http://localhost:{LOCAL_PORT}/predict"

try:
    response = requests.post(url, json=TEST_CLIENT_DATA, timeout=5)
    result = response.json()
    
    print("\n" + "="*50)
    print("🎯 QUESTION 1 ANSWER")
    print("="*50)
    print(f"Input data: {json.dumps(TEST_CLIENT_DATA, indent=2)}")
    print(f"\nPrediction result: {json.dumps(result, indent=2)}")
    
    if 'conversion_probability' in result:
        prob = result['conversion_probability']
        print(f"\n✅ Probability: {prob:.3f}")
        print(f"\nSelect the closest option from the homework choices.")
    else:
        print(f"\n⚠️ Unexpected response format. Full response: {result}")
        
except Exception as e:
    print(f"❌ Error testing endpoint: {e}")
    print("\nCheck if container is running:")
    !docker logs zoomcamp-test --tail 20


🎯 QUESTION 1 ANSWER
Input data: {
  "job": "management",
  "duration": 400,
  "poutcome": "success"
}

Prediction result: {
  "conversion_probability": 0.49999999999842815,
  "conversion": false
}

✅ Probability: 0.500

Select the closest option from the homework choices.


In [7]:
# Clean up - stop the container after testing
# Uncomment when done with local testing
# !docker stop zoomcamp-test
# !docker rm zoomcamp-test

## Question 2: Kind Version

**Task:** Check the installed version of `kind`

In [8]:
# Check kind version
!kind --version

print("\n" + "="*50)
print("🎯 QUESTION 2 ANSWER")
print("="*50)
print("Record the version number shown above.")

kind version 0.30.0

🎯 QUESTION 2 ANSWER
Record the version number shown above.


## Question 3: Kubernetes Concepts

**Task:** What is the smallest deployable unit of computing in Kubernetes?

**Options:**
- Node
- Pod ✅ (Expected answer)
- Deployment
- Service

In [9]:
print("="*50)
print("🎯 QUESTION 3 ANSWER")
print("="*50)
print("\nThe smallest deployable unit in Kubernetes is: Pod")
print("\nExplanation:")
print("- A Pod is a group of one or more containers")
print("- Pods are the atomic unit of scheduling in Kubernetes")
print("- Deployments and Services manage Pods")
print("- Nodes are physical/virtual machines that run Pods")

🎯 QUESTION 3 ANSWER

The smallest deployable unit in Kubernetes is: Pod

Explanation:
- A Pod is a group of one or more containers
- Pods are the atomic unit of scheduling in Kubernetes
- Deployments and Services manage Pods
- Nodes are physical/virtual machines that run Pods


## Kubernetes Cluster Setup

In [10]:
# Check if kubectl is installed
!kubectl version --client

Client Version: v1.34.1
Kustomize Version: v5.7.1


In [11]:
# Delete existing cluster if it exists
!kind delete cluster --name {CLUSTER_NAME} 2>/dev/null || true

In [12]:
# Create a new kind cluster
!kind create cluster --name {CLUSTER_NAME}

print("\nWaiting for cluster to be ready...")
!kubectl cluster-info --context kind-{CLUSTER_NAME}

Creating cluster "kind" ...
 ✓ Ensuring node image (kindest/node:v1.34.0) 🖼
 ✓ Preparing nodes 📦 7l
 ✓ Writing configuration 📜7l
 ✓ Starting control-plane 🕹️7l
 ✓ Installing CNI 🔌7l
 ✓ Installing StorageClass 💾7l
Set kubectl context to "kind-kind"
You can now use your cluster with:

kubectl cluster-info --context kind-kind

Have a question, bug, or feature request? Let us know! https://kind.sigs.k8s.io/#community 🙂

Waiting for cluster to be ready...
Kubernetes control plane is running at https://127.0.0.1:54499
CoreDNS is running at https://127.0.0.1:54499/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.


## Question 4: Service Types

**Task:** What service type is running after cluster creation?

In [13]:
# List all services
!kubectl get services --all-namespaces

print("\n" + "="*50)
print("🎯 QUESTION 4 ANSWER")
print("="*50)
print("Look at the TYPE column for the kubernetes service above.")
print("Expected answer: ClusterIP")

NAMESPACE     NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)                  AGE
default       kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP                  3s
kube-system   kube-dns     ClusterIP   10.96.0.10   <none>        53/UDP,53/TCP,9153/TCP   3s

🎯 QUESTION 4 ANSWER
Look at the TYPE column for the kubernetes service above.
Expected answer: ClusterIP


## Question 5: Load Docker Image to Kind

**Task:** What command loads a Docker image into the kind cluster?

In [14]:
print("="*50)
print("🎯 QUESTION 5 ANSWER")
print("="*50)
print("\nCommand to load Docker image into kind:")
print(f"kind load docker-image {DOCKER_IMAGE_FULL} --name {CLUSTER_NAME}")
print("\nExecuting the command...\n")

# Actually load the image
!kind load docker-image {DOCKER_IMAGE_FULL} --name {CLUSTER_NAME}

🎯 QUESTION 5 ANSWER

Command to load Docker image into kind:
kind load docker-image zoomcamp-model:3.13.10-hw10 --name kind

Executing the command...

Image: "zoomcamp-model:3.13.10-hw10" with ID "sha256:4ad55623c4703028fffc6a072e06ff67e021be26a2783940fd88420fee756348" not yet present on node "kind-control-plane", loading...


## Question 6: Create Deployment

**Task:** Create a deployment with the specified configuration

In [15]:
# Apply the deployment from the YAML file
!kubectl apply -f deployment.yaml

print("\nWaiting for deployment to be ready...")
!kubectl wait --for=condition=available --timeout=60s deployment/{DEPLOYMENT_NAME}

print("\nDeployment status:")
!kubectl get deployments
!kubectl get pods

deployment.apps/subscription created

Waiting for deployment to be ready...
deployment.apps/subscription condition met

Deployment status:
NAME           READY   UP-TO-DATE   AVAILABLE   AGE
subscription   1/1     1            1           12s
NAME                           READY   STATUS    RESTARTS   AGE
subscription-568c5d8cf-lw2m7   1/1     Running   0          12s


In [16]:
# Describe the deployment to verify configuration
!kubectl describe deployment {DEPLOYMENT_NAME}

print("\n" + "="*50)
print("🎯 QUESTION 6 ANSWER")
print("="*50)
print("Check the deployment.yaml file for the containerPort value.")
print(f"Expected containerPort: {TARGET_PORT}")

Name:                   subscription
Namespace:              default
CreationTimestamp:      Thu, 11 Dec 2025 16:28:02 +0100
Labels:                 <none>
Annotations:            deployment.kubernetes.io/revision: 1
Selector:               app=subscription
Replicas:               1 desired | 1 updated | 1 total | 1 available | 0 unavailable
StrategyType:           RollingUpdate
MinReadySeconds:        0
RollingUpdateStrategy:  25% max unavailable, 25% max surge
Pod Template:
  Labels:  app=subscription
  Containers:
   subscription:
    Image:      zoomcamp-model:3.13.10-hw10
    Port:       9696/TCP
    Host Port:  0/TCP
    Limits:
      cpu:     200m
      memory:  128Mi
    Requests:
      cpu:         100m
      memory:      64Mi
    Environment:   <none>
    Mounts:        <none>
  Volumes:         <none>
  Node-Selectors:  <none>
  Tolerations:     <none>
Conditions:
  Type           Status  Reason
  ----           ------  ------
  Available      True    MinimumReplicasAvailabl

## Question 7: Create LoadBalancer Service

**Task:** Create a LoadBalancer service and identify the selector value

In [17]:
# Apply the service from the YAML file
!kubectl apply -f service.yaml

print("\nService created:")
!kubectl get services

print("\nService details:")
!kubectl describe service {SERVICE_NAME}

service/subscription created

Service created:
NAME           TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)        AGE
kubernetes     ClusterIP      10.96.0.1       <none>        443/TCP        24s
subscription   LoadBalancer   10.96.107.249   <pending>     80:30180/TCP   0s

Service details:
Name:                     subscription
Namespace:                default
Labels:                   <none>
Annotations:              <none>
Selector:                 app=subscription
Type:                     LoadBalancer
IP Family Policy:         SingleStack
IP Families:              IPv4
IP:                       10.96.107.249
IPs:                      10.96.107.249
Port:                     <unset>  80/TCP
TargetPort:               9696/TCP
NodePort:                 <unset>  30180/TCP
Endpoints:                10.244.0.3:9696
Session Affinity:         None
External Traffic Policy:  Cluster
Internal Traffic Policy:  Cluster
Events:                   <none>


In [18]:
print("="*50)
print("🎯 QUESTION 7 ANSWER")
print("="*50)
print("Check the service.yaml file for the selector app value.")
print("The selector should match the label in the deployment.")
print(f"\nExpected selector: app: subscription")

🎯 QUESTION 7 ANSWER
Check the service.yaml file for the selector app value.
The selector should match the label in the deployment.

Expected selector: app: subscription


## Test the Service with Port Forwarding

In [19]:
# Note: Port forwarding needs to run in background
# You may need to run this in a separate terminal:
print(f"Run this command in a separate terminal:")
print(f"kubectl port-forward service/{SERVICE_NAME} {LOCAL_PORT}:{SERVICE_PORT}")
print("\nThen test with the cell below.")

Run this command in a separate terminal:
kubectl port-forward service/subscription 9696:80

Then test with the cell below.


In [20]:
# Test the service through port-forward
# Make sure port-forward is running before executing this cell

import requests
import json

url = f"http://localhost:{LOCAL_PORT}/predict"

try:
    response = requests.post(url, json=TEST_CLIENT_DATA, timeout=5)
    result = response.json()
    
    print("\n✅ Service is responding!")
    print(f"Result: {json.dumps(result, indent=2)}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure port-forward is running!")


✅ Service is responding!
Result: {
  "conversion_probability": 0.49999999999842815,
  "conversion": false
}


## Question 8: Horizontal Pod Autoscaler (Optional)

**Task:** Configure HPA and perform load testing to see maximum replicas

In [21]:
# Install metrics-server for HPA (required for kind)
print("Installing metrics-server...")
!kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

# Patch metrics-server for kind (insecure TLS)
!kubectl patch deployment metrics-server -n kube-system --type='json' -p='[{"op": "add", "path": "/spec/template/spec/containers/0/args/-", "value": "--kubelet-insecure-tls"}]'

print("\nWaiting for metrics-server to be ready...")
import time
time.sleep(10)
!kubectl wait --for=condition=available --timeout=60s deployment/metrics-server -n kube-system

Installing metrics-server...
serviceaccount/metrics-server created
clusterrole.rbac.authorization.k8s.io/system:aggregated-metrics-reader created
clusterrole.rbac.authorization.k8s.io/system:metrics-server created
rolebinding.rbac.authorization.k8s.io/metrics-server-auth-reader created
clusterrolebinding.rbac.authorization.k8s.io/metrics-server:system:auth-delegator created
clusterrolebinding.rbac.authorization.k8s.io/system:metrics-server created
service/metrics-server created
deployment.apps/metrics-server created
apiservice.apiregistration.k8s.io/v1beta1.metrics.k8s.io created
deployment.apps/metrics-server patched

Waiting for metrics-server to be ready...
deployment.apps/metrics-server condition met


In [22]:
# Apply HPA configuration
!kubectl apply -f hpa.yaml

print("\nHPA status:")
!kubectl get hpa

horizontalpodautoscaler.autoscaling/subscription-hpa created

HPA status:
NAME               REFERENCE                 TARGETS              MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: <unknown>/20%   1         3         0          0s


In [23]:
# Run load test
print(f"Starting load test with {LOAD_TEST_REQUESTS} requests...")
print("This will take a few minutes.\n")

import requests
import time

url = f"http://localhost:{LOCAL_PORT}/predict"
successful_requests = 0
failed_requests = 0

for i in range(LOAD_TEST_REQUESTS):
    try:
        response = requests.post(url, json=TEST_CLIENT_DATA, timeout=5)
        if response.status_code == 200:
            successful_requests += 1
        else:
            failed_requests += 1
    except Exception as e:
        failed_requests += 1
    
    if i % 100 == 0:
        print(f"Progress: {i}/{LOAD_TEST_REQUESTS} requests sent")
    
    time.sleep(LOAD_TEST_DELAY)

print(f"\n✅ Load test complete!")
print(f"Successful: {successful_requests}")
print(f"Failed: {failed_requests}")

Starting load test with 1000 requests...
This will take a few minutes.

Progress: 0/1000 requests sent
Progress: 100/1000 requests sent
Progress: 200/1000 requests sent
Progress: 300/1000 requests sent
Progress: 400/1000 requests sent
Progress: 500/1000 requests sent
Progress: 600/1000 requests sent
Progress: 700/1000 requests sent
Progress: 800/1000 requests sent
Progress: 900/1000 requests sent

✅ Load test complete!
Successful: 1000
Failed: 0


In [24]:
# Monitor HPA and pods during/after load test
print("HPA Status:")
!kubectl get hpa

print("\nPod Status:")
!kubectl get pods

print("\nDeployment Replicas:")
!kubectl get deployment {DEPLOYMENT_NAME}

print("\n" + "="*50)
print("🎯 QUESTION 8 ANSWER")
print("="*50)
print("Count the number of READY pods above.")
print("This is the maximum number of replicas achieved during load test.")

HPA Status:
NAME               REFERENCE                 TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 3%/20%   1         3         1          19s

Pod Status:
NAME                           READY   STATUS    RESTARTS   AGE
subscription-568c5d8cf-lw2m7   1/1     Running   0          69s

Deployment Replicas:
NAME           READY   UP-TO-DATE   AVAILABLE   AGE
subscription   1/1     1            1           69s

🎯 QUESTION 8 ANSWER
Count the number of READY pods above.
This is the maximum number of replicas achieved during load test.


In [25]:
# Watch HPA over time to see scaling behavior
# Note: This will run for 2 minutes
print("Monitoring HPA for 2 minutes...\n")

import time
for i in range(24):  # 24 * 5 seconds = 2 minutes
    !kubectl get hpa {HPA_NAME}
    !kubectl get pods | grep {DEPLOYMENT_NAME}
    print(f"\nTime: {i*5} seconds\n")
    time.sleep(5)

Monitoring HPA for 2 minutes...

NAME               REFERENCE                 TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 3%/20%   1         3         1          19s
subscription-568c5d8cf-lw2m7   1/1     Running   0          70s

Time: 0 seconds

NAME               REFERENCE                 TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 3%/20%   1         3         1          25s
subscription-568c5d8cf-lw2m7   1/1     Running   0          75s

Time: 5 seconds

NAME               REFERENCE                 TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 3%/20%   1         3         1          31s
subscription-568c5d8cf-lw2m7   1/1     Running   0          81s

Time: 10 seconds

NAME               REFERENCE                 TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 3%/20%   1  

## Cleanup

Run these cells when you're done to clean up resources

In [26]:
# Delete Kubernetes resources
!kubectl delete hpa {HPA_NAME}
!kubectl delete service {SERVICE_NAME}
!kubectl delete deployment {DEPLOYMENT_NAME}

horizontalpodautoscaler.autoscaling "subscription-hpa" deleted from default namespace
service "subscription" deleted from default namespace
deployment.apps "subscription" deleted from default namespace


In [27]:
# Delete the kind cluster
!kind delete cluster --name {CLUSTER_NAME}

Deleting cluster "kind" ...
Deleted nodes: ["kind-control-plane"]


In [28]:
# Stop and remove Docker container
!docker stop zoomcamp-test 2>/dev/null || true
!docker rm zoomcamp-test 2>/dev/null || true

zoomcamp-test
zoomcamp-test


## Summary of Answers

Run this cell to see all answers in one place:

In [29]:
print("="*60)
print("     CHAPTER 10 KUBERNETES HOMEWORK - ANSWER SUMMARY")
print("="*60)
print("\nQ1: Model Probability")
print("    → Run the local Docker test to get the probability value")
print("\nQ2: Kind Version")
print("    → Run 'kind --version' to get the version")
print("\nQ3: Smallest Kubernetes Unit")
print("    → Answer: Pod")
print("\nQ4: Default Service Type")
print("    → Answer: ClusterIP")
print("\nQ5: Load Image to Kind")
print("    → Command: kind load docker-image")
print("\nQ6: Container Port")
print("    → Check deployment.yaml for containerPort value")
print("\nQ7: Service Selector")
print("    → Check service.yaml for selector app value")
print("\nQ8: Maximum Replicas (Optional)")
print("    → Count pods after load test")
print("\n" + "="*60)

     CHAPTER 10 KUBERNETES HOMEWORK - ANSWER SUMMARY

Q1: Model Probability
    → Run the local Docker test to get the probability value

Q2: Kind Version
    → Run 'kind --version' to get the version

Q3: Smallest Kubernetes Unit
    → Answer: Pod

Q4: Default Service Type
    → Answer: ClusterIP

Q5: Load Image to Kind
    → Command: kind load docker-image

Q6: Container Port
    → Check deployment.yaml for containerPort value

Q7: Service Selector
    → Check service.yaml for selector app value

Q8: Maximum Replicas (Optional)
    → Count pods after load test

